# 第15章 屋内 Wi-Fi の電波伝搬解析

書籍『Sionna RTによる電波伝搬レイトレーシング入門』第15章のコードである。
セルは本文の掲載順に並べてあり、コードセルの中身は本文に印刷したものと同じである。

先頭の「準備」セルだけは本文に印刷していない。本文のコードが前提にしている
import とシーンの読み込みをここで済ませてある。準備セルを実行すれば、以降は
上から順に実行できる。

ただし、本文が説明のために示している断片（自分で作ったシーンのパスや、その
シーンには無い物体名を使う例）はそのままでは動かない。該当するセルの前に注記を
置いてある。

## 準備

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import sionna.rt as rt
from sionna.rt import (load_scene, scene as scene_assets,
                       Transmitter, Receiver, PlanarArray,
                       PathSolver, RadioMapSolver, Camera, RadioMaterial)

# 付属の 2 部屋アパートシーン（このリポジトリの scenes/ にある）
scene = load_scene('scenes/indoor_apartment/scene.xml')
scene.frequency = 5e9
scene.tx_array = PlanarArray(num_rows=1, num_cols=1,
                             pattern='iso', polarization='V')
scene.rx_array = PlanarArray(num_rows=1, num_cols=1,
                             pattern='iso', polarization='V')

rmsolver = RadioMapSolver()
rss_dbm = None   # 電波マップを計算したあとに代入する

## 4. アクセスポイントの配置

In [ ]:
# 自作シーンを読み込む
scene = load_scene('scenes/indoor_apartment/scene.xml')
scene.frequency = 5e9

# AP 1 台案 (左室)
ap1 = Transmitter('ap1', position=[-3.5, 0.0, 2.0],
                  power_dbm=20.0)
# AP 2 台案 (右室にも追加)
ap2 = Transmitter('ap2', position=[3.5, 0.0, 2.0],
                  power_dbm=20.0)

## 5. 受信電力マップを作る

In [ ]:
rm = rmsolver(scene, max_depth=4,
              cell_size=[0.25, 0.25],
              center=[0.0, 0.0, 1.0],   # 床から 1.0 m の受信面
              samples_per_tx=3_000_000)

## 6. デッドスポットを探す

In [ ]:
dead = rss_dbm < -70.0    # Wi-Fi の実用しきい値
plt.imshow(dead, cmap='Reds')